# Solucion 6: respuestas a las 5 preguntas de Exercises Python

Origen: `00. Intro/Exersices_Python.ipynb`.


In [ ]:
import numpy as np
from scipy import optimize


## Pregunta 1. Galaxia espiral

- `contourf` con niveles logaritmicos deja ver al mismo tiempo el bulbo brillante y la parte tenue del disco.
- El modelo representa una densidad estelar proyectada con bulbo, disco y barra.
- Para agregar brazos espirales, conviene modular el disco con una fase angular.


In [ ]:
def gaussian_spiral_galaxy(x, y, bulge_params, disk_params, bar_params,
                           arm_strength=0.3, arm_count=2, pitch_angle=15*np.pi/180):
    return "firma sugerida para extender el modelo"


gaussian_spiral_galaxy(0, 0, None, None, None)


## Pregunta 2. Grafica de clase del 14 de febrero


In [ ]:
phi_bogota = 4.6481
phi_medellin = 6.25
delta_betelgeuse = 7.4

hmax_bogota = 90 - abs(phi_bogota - delta_betelgeuse)
hmax_medellin = 90 - abs(phi_medellin - delta_betelgeuse)

print("hmax Bogota:", hmax_bogota)
print("hmax Medellin:", hmax_medellin)


La conclusion es directa: Betelgeuse si supera con amplitud los `30 grados`, asi que la frase del notebook original no queda bien.


## Pregunta 3. Orbit fit


In [ ]:
def kepler_orbit(t, a, e, i, Omega, omega, T0):
    M = 2 * np.pi * (t - T0) / 365.25
    E = M + e * np.sin(M)
    nu = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    r = a * (1 - e**2) / (1 + e * np.cos(nu))
    delta_ra = r * np.cos(omega + nu) * np.cos(i) / 1000
    delta_dec = r * np.sin(omega + nu) / 1000
    return delta_ra, delta_dec


def residuals(params, t, ra, dec):
    ra_fit, dec_fit = kepler_orbit(t, *params)
    return np.concatenate([ra - ra_fit, dec - dec_fit])


rng = np.random.default_rng(42)
t_obs = np.linspace(0, 5, 50)
true_params = [100, 0.3, np.pi / 4, 0, np.pi / 3, 0.1]
ra_true, dec_true = kepler_orbit(t_obs, *true_params)
ra_obs = ra_true + 2 * rng.normal(size=50)
dec_obs = dec_true + 2 * rng.normal(size=50)

p0 = [80, 0.2, np.pi / 3, 0, np.pi / 4, 0]
result = optimize.least_squares(residuals, p0, args=(t_obs, ra_obs, dec_obs))

res = residuals(result.x, t_obs, ra_obs, dec_obs)
sigma = 2.0
chi2 = np.sum((res / sigma) ** 2)
dof = len(res) - len(result.x)
reduced_chi2 = chi2 / dof

print("e ajustada:", result.x[1])
print("chi^2 reducido:", reduced_chi2)


- `e = 0` significa una orbita circular.
- Dos mejoras razonables: mejores incertidumbres/cadencia y una solucion mas fiel de la ecuacion de Kepler.
- El `chi^2` reducido sale del orden de `0.86`, compatible con un ajuste dominado por ruido.


## Pregunta 4. Altair

- Al seleccionar la secuencia principal, el histograma de color se concentra en valores medios de `BP-RP`.
- El tamano del marcador se amarra a la paralaje para dar mas peso visual a las estrellas cercanas.
- El panel acumulado puede agregarse con el siguiente bloque.


In [ ]:
altair_snippet = '''
cumulative_abs_mag = base.transform_filter(
    brush
).transform_window(
    rank="rank(abs_G)",
    sort=[alt.SortField("abs_G")]
).transform_joinaggregate(
    total="count()"
).transform_calculate(
    cdf="datum.rank / datum.total"
).mark_line(color="darkorange").encode(
    x=alt.X("abs_G:Q", title="Absolute Magnitude"),
    y=alt.Y("cdf:Q", title="Cumulative Fraction")
)
'''

print(altair_snippet)


## Pregunta 5. SunPy + integracion

- Un pico cerca de `15 min` se parece mas a una oscilacion coronal o magnetoacustica lenta que al p-mode clasico de 5 minutos.
- `submap` se usa para no diluir la senal de la region activa con el disco solar completo.
- Si la curva realmente oscila, un seno amortiguado deberia ajustar mejor que un box transit.


In [ ]:
def damped_sine(t, A, tau, period, phi, c):
    return c + A * np.exp(-t / tau) * np.sin(2 * np.pi * t / period + phi)


def residuals_damped(params, t, flux):
    return flux - damped_sine(t, *params)


print("Modelo propuesto listo para comparar contra el box transit.")
